In [48]:
# load documents
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [49]:
print(len(documents))
documents[0]

72


{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [50]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [51]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [52]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [53]:
# filter for 3 lessons
three_filtered_filenames = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md"
]

three_filtered_documents = list(filter(
    lambda d: d['filename'] in three_filtered_filenames, documents
))

In [54]:
print(len(three_filtered_documents))
three_filtered_documents

3


[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

In [55]:
from evaluation_utils import llm_structured_retry
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    output, usage = llm_structured_retry(
        client=openai_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model="gpt-5.4-mini",
    )

    results = []

    for q in output.questions:
        results.append({
            "question": q,
            "filename": doc["filename"]
        })

    return results, usage

In [56]:
# run llm call in parallel and store the set of questions for all 3 docs
from evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=4) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/72 [00:00<?, ?it/s]

In [57]:
results[0]

([{'question': 'What is the basic idea behind RAG, and how does it help with answers that the model doesn’t already know?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'Why does this course treat the LLM like a black box instead of teaching how the model itself works?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What are the main limits of an LLM that make retrieval useful in the first place?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'What will be built in this module’s first part, and what steps does it include?',
   'filename': '01-agentic-rag/lessons/01-intro.md'},
  {'question': 'How is the second part of the module different from the first part in how the system chooses what to search?',
   'filename': '01-agentic-rag/lessons/01-intro.md'}],
 ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=119, output_tokens_details=OutputT

In [58]:
ground_truths = []
usages = []

for questions, usage in results:
    ground_truths.extend(questions)
    usages.append(usage)

In [59]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.11349

In [60]:
ground_truths[0]

{'question': 'What is the basic idea behind RAG, and how does it help with answers that the model doesn’t already know?',
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [61]:
from evaluation_utils import map_progress
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=4) as pool:
    results_for_three = map_progress(pool, three_filtered_documents, generate_ground_truth)

  0%|          | 0/3 [00:00<?, ?it/s]

In [62]:
ground_truths_for_three = []
usages_for_three = []

for questions, usage in results_for_three:
    ground_truths_for_three.extend(questions)
    usages_for_three.append(usage)

In [63]:
from evaluation_utils import calc_total_price

print(calc_total_price(usages_for_three))
print(len(ground_truths_for_three))
print(ground_truths_for_three[0])

0.00444825
15
{'question': 'What problem does retrieval-augmented generation solve that a plain LLM cannot handle well?', 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [64]:
total_input_tokens = 0

for u in usages_for_three:
    total_input_tokens += u.input_tokens

print(total_input_tokens / 3)

1353.0


In [65]:
# chunk documents
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [66]:
print(chunks[0])
len(chunks)

{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone

295

In [67]:
# setup embedder
from embedder import Embedder

embed = Embedder()

In [68]:
# setup texts for embedding
texts = [chunk['content'] for chunk in chunks]

In [69]:
print(texts[0])
print(len(texts))

# Introduction

Video: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In this module, we'll build a working Retrieval-Augmented
Generation (RAG) system from scratch, step by step.

We write everything in plain Python. We build a small search index by
hand and call the LLM ourselves. I want you to see every piece first.
That way you know what a framework does for you before you reach for
one.

Places where you can find me:

- [My substack](https://alexeyondata.substack.com/)
- [LinkedIn](https://www.linkedin.com/in/agrigorev/)
- [X](https://x.com/Al_Grigor)

## LLMs

An LLM (Large Language Model) is a neural network trained on massive
amounts of text. Given a prompt, it generates a continuation - a
plausible next piece of text.

Think of your phone. When you type "how are" in WhatsApp, it suggests
"you" as the next word. "How are you" is the most common continuation.
Your phone uses a simple language model for that. It predicts 

In [70]:
# embed texts

from tqdm.auto import tqdm
import numpy as np

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)


  0%|          | 0/6 [00:00<?, ?it/s]

In [71]:
# index vectors into DB
from minsearch import VectorSearch

vIndex = VectorSearch(keyword_fields=['filename'])
vIndex.fit(X, chunks)

In [72]:
# index text into DB
from minsearch import Index

tIndex = Index(
    keyword_fields=['filename'],
    text_fields=['content']
)

tIndex.fit(chunks)

In [73]:
# hybrid search
# Each search produces its own ranked list, so we need a way to combine them into one. 
# We use Reciprocal Rank Fusion (RRF). 
# It ignores the raw scores from each method, which live on different scales and aren't directly comparable. 
# Instead, it looks only at the position of each document in each list.

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [85]:
def text_search(query, num_results=5):
    return tIndex.search(query=query, num_results=num_results)

def vector_search(query, num_results=5):
    vector_query = embed.encode(query)
    return vIndex.search(query_vector=vector_query, num_results=num_results)

In [75]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
# load ground truth data
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [82]:
q = ground_truth[0]["question"]

In [97]:
print(q)
print(ground_truth[0])
print(len(ground_truth))

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?", 'filename': '01-agentic-rag/lessons/01-intro.md'}
360


In [86]:
text_res = text_search(q)

In [87]:
text_res[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [88]:
vector_res = vector_search(q)

In [89]:
vector_res[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [115]:
def compute_relevance(q, search_function, **kwargs):
    filename = q["filename"]
    results = search_function(query=q["question"], **kwargs)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == filename))

    return relevance

In [114]:
def compute_relevance_total(ground_truth, search_function, **kwargs):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function, **kwargs)
        relevance_total.append(relevance)

    return relevance_total

In [104]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [117]:
def evaluate(ground_truth, search_function, **kwargs):
    relevance_total = compute_relevance_total(ground_truth, search_function, **kwargs)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [106]:
result_for_vector_search = evaluate(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [107]:
result_for_vector_search

{'hit_rate': 0.725, 'mrr': 0.5486111111111112}

In [108]:
result_for_text_search = evaluate(ground_truth, text_search)

  0%|          | 0/360 [00:00<?, ?it/s]

In [110]:
result_for_text_search

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}

In [120]:
#  When evaluating hybrid_search for k values 1, 50, 100, and 200, which k gives the best MRR?

hybrid_search_results_by_k = {
    1: {},
    50: {},
    100: {},
    200: {}
}

for k in hybrid_search_results_by_k.keys():
    result = evaluate(ground_truth=ground_truth, search_function=hybrid_search, k=k)
    hybrid_search_results_by_k[k] = result


  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

In [121]:
hybrid_search_results_by_k

{1: {'hit_rate': 0.8388888888888889, 'mrr': 0.6481944444444449},
 50: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 100: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667},
 200: {'hit_rate': 0.8361111111111111, 'mrr': 0.637916666666667}}